# InsightQMC checkpoint 参数统计

运行下面唯一的代码单元，即可从 checkpoint 读取并分类罗列所有可训练参数。`CHECKPOINT_PATH = None` 时自动选择 `outputs/` 下最新的 `last.pkl`。

In [1]:
from pathlib import Path
import pickle
from collections.abc import Mapping

import numpy as np

# 填写绝对路径或相对 InsightQMC/ 的路径；None 表示自动读取最新 checkpoint。
CHECKPOINT_PATH = None

project_root = Path.cwd()
if not (project_root / "networks.py").exists() and (project_root / "InsightQMC" / "networks.py").exists():
    project_root = project_root / "InsightQMC"

if CHECKPOINT_PATH is None:
    candidates = list((project_root / "outputs").rglob("last.pkl"))
    if not candidates:
        raise FileNotFoundError(f"在 {project_root / 'outputs'} 下没有找到 last.pkl")
    ckpt_path = max(candidates, key=lambda path: path.stat().st_mtime)
else:
    ckpt_path = Path(CHECKPOINT_PATH).expanduser()
    if not ckpt_path.is_absolute():
        ckpt_path = project_root / ckpt_path
if not ckpt_path.is_file():
    raise FileNotFoundError(f"checkpoint 不存在: {ckpt_path}")

with ckpt_path.open("rb") as handle:
    ckpt = pickle.load(handle)

CATEGORY_NAMES = {
    "mkan": "MKAN/KAN 主体网络",
    "orbital_head": "轨道输出头",
    "envelope": "轨道包络",
    "jastrow_ee": "电子-电子 Jastrow",
    "jastrow_en": "电子-原子 Jastrow",
    "det_weights": "多行列式组合权重",
}

def iter_parameter_leaves(value, path=()):
    """递归产生（参数路径, 数组），只统计带 shape 的数组叶子。"""
    if hasattr(value, "shape"):
        yield path, value
    elif isinstance(value, Mapping) or hasattr(value, "items"):
        for key, child in value.items():
            yield from iter_parameter_leaves(child, path + (str(key),))
    elif isinstance(value, (list, tuple)):
        for index, child in enumerate(value):
            yield from iter_parameter_leaves(child, path + (str(index),))

params = ckpt.get("params")
if not isinstance(params, Mapping):
    raise TypeError("checkpoint 中没有可识别的 params 参数树")

details = []
category_counts = {}
for top_key, subtree in params.items():
    category = CATEGORY_NAMES.get(str(top_key), f"其他: {top_key}")
    category_total = 0
    for path, leaf in iter_parameter_leaves(subtree, (str(top_key),)):
        shape = tuple(int(dim) for dim in leaf.shape)
        count = int(np.prod(shape, dtype=np.int64))
        category_total += count
        details.append((category, ".".join(path), shape, str(leaf.dtype), count))
    category_counts[category] = category_total

total = sum(category_counts.values())
print(f"checkpoint: {ckpt_path}")
print(f"stage/step: {ckpt.get('stage')} / {ckpt.get('step')}")
print("\n== 参数类别汇总 ==")
for category, count in sorted(category_counts.items(), key=lambda item: item[1], reverse=True):
    percentage = 100.0 * count / total if total else 0.0
    print(f"{category:<24} {count:>12,}  ({percentage:6.2f}%)")
print(f"{'总计':<24} {total:>12,}  ({100.0 if total else 0.0:.2f}%)")

print("\n== 逐参数明细 ==")
for category, path, shape, dtype, count in details:
    print(f"{category:<24} {count:>10,}  shape={str(shape):<20} dtype={dtype:<10} {path}")

print("\n说明：Slater M 矩阵由网络轨道输出和 envelope 在前向计算中动态构造，")
print("它不是 checkpoint 中独立保存的可训练参数，因此不会作为单独类别出现。")

checkpoint: /vepfs-mlp2/c20250516/250504030/jing/InsightQMC/outputs/C/basis_test/chebyshev/20260812_033025/checkpoints/last.pkl
stage/step: train / 30000

== 参数类别汇总 ==
MKAN/KAN 主体网络                  33,296  ( 90.44%)
轨道输出头                           3,168  (  8.61%)
轨道包络                              336  (  0.91%)
电子-电子 Jastrow                      10  (  0.03%)
多行列式组合权重                            4  (  0.01%)
总计                             36,814  (100.00%)

== 逐参数明细 ==
多行列式组合权重                          4  shape=(4, 1)               dtype=float32    det_weights
轨道包络                             64  shape=(1, 4, 16)           dtype=float32    envelope.0.angular_coeff
轨道包络                             16  shape=(1, 16)              dtype=float32    envelope.0.pi
轨道包络                            144  shape=(1, 16, 3, 3)        dtype=float32    envelope.0.sigma
轨道包络                             32  shape=(1, 4, 8)            dtype=float32    envelope.1.angular_coeff
轨道包络                       